# NB03: UniProt-Native Evidence (Tier 1)

**Purpose**: Extract protein-EC mappings from the three UniProt-native evidence channels:
1. UniProt EC cross-references (`identifier WHERE db='EC'`, ~38.6M rows)
2. UniProt BRENDA cross-references (`identifier WHERE db='BRENDA'`, ~38.6K rows)
3. Rhea catalytic activity from `comment_xml` (330K rows) - extract co-annotated EC numbers

All EC numbers are checked against the EC-reaction bridge (NB02) to confirm they
reach mass-balanced reactions, but the full protein x reaction expansion is deferred
to the integration notebook (NB06) to manage memory.

**Note**: `identifier` and `comment_xml` use `entity_id` format `uniprot:ACCESSION`.

**Output**: `uniprot_native_protein_ec.parquet` - protein-EC pairs with channel tags.

**Requires**: BERDL JupyterHub (Spark session), NB02 bridge tables

In [1]:
import os, re
import pandas as pd

try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
except ImportError:
    from get_spark_session import get_spark_session

spark = get_spark_session()
spark.sql("SET spark.sql.autoBroadcastJoinThreshold = -1")

DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

ec_bridge = pd.read_parquet(f'{DATA_DIR}/ec_to_reaction.parquet')
print(f'EC bridge: {len(ec_bridge):,} mappings, {ec_bridge.ec.nunique():,} ECs, {ec_bridge.rxn_bare.nunique():,} reactions')

bridge_ecs = set(ec_bridge['ec'])
print(f'Spark session ready.')

EC bridge: 22,823 mappings, 6,100 ECs, 18,660 reactions
Spark session ready.


## 1. UniProt EC Cross-References

Query `refdata_uniprot.identifier` for `db='EC'`. Filter to bridge ECs on Spark side.

In [2]:
uniprot_ec_spark = spark.sql("""
    SELECT entity_id, xref AS ec
    FROM refdata_uniprot.identifier
    WHERE db = 'EC'
""")

total_ec_rows = uniprot_ec_spark.count()
total_ec_proteins = uniprot_ec_spark.select('entity_id').distinct().count()
total_ec_unique = uniprot_ec_spark.select('ec').distinct().count()
print(f'UniProt EC xrefs (full): {total_ec_rows:,}')
print(f'  Unique proteins: {total_ec_proteins:,}')
print(f'  Unique ECs:      {total_ec_unique:,}')

UniProt EC xrefs (full): 38,563,959
  Unique proteins: 34,906,907
  Unique ECs:      6,762


In [3]:
uniprot_ec_df = uniprot_ec_spark.filter(
    uniprot_ec_spark.ec.isin(list(bridge_ecs))
).toPandas()

uniprot_ec_df['protein'] = uniprot_ec_df['entity_id'].str.replace('uniprot:', '', regex=False)
uniprot_ec_pairs = uniprot_ec_df[['protein', 'ec']].drop_duplicates()
uniprot_ec_pairs['channel'] = 'uniprot_ec'

print(f'After filtering to bridge ECs:')
print(f'  Protein-EC pairs: {len(uniprot_ec_pairs):,}')
print(f'  Unique proteins:  {uniprot_ec_pairs.protein.nunique():,}')
print(f'  Unique ECs:       {uniprot_ec_pairs.ec.nunique():,}')

del uniprot_ec_df

After filtering to bridge ECs:
  Protein-EC pairs: 27,630,112


  Unique proteins:  26,543,162


  Unique ECs:       4,938


## 2. UniProt BRENDA Cross-References

BRENDA entries are EC numbers. Much smaller set (~38.6K rows) but curated.

In [4]:
brenda_df = spark.sql("""
    SELECT entity_id, xref AS ec
    FROM refdata_uniprot.identifier
    WHERE db = 'BRENDA'
""").toPandas()

brenda_df['protein'] = brenda_df['entity_id'].str.replace('uniprot:', '', regex=False)

print(f'BRENDA xrefs: {len(brenda_df):,}')
print(f'  Unique proteins: {brenda_df.protein.nunique():,}')
print(f'  Unique ECs:      {brenda_df.ec.nunique():,}')

brenda_ecs_in_bridge = set(brenda_df['ec']) & bridge_ecs
print(f'  ECs matching bridge: {len(brenda_ecs_in_bridge):,}')

brenda_pairs = brenda_df[brenda_df['ec'].isin(bridge_ecs)][['protein', 'ec']].drop_duplicates()
brenda_pairs['channel'] = 'brenda'

print(f'\nBRENDA protein-EC pairs (bridge-matched):')
print(f'  Pairs:    {len(brenda_pairs):,}')
print(f'  Proteins: {brenda_pairs.protein.nunique():,}')
print(f'  ECs:      {brenda_pairs.ec.nunique():,}')

del brenda_df

BRENDA xrefs: 38,623
  Unique proteins: 35,082
  Unique ECs:      5,549
  ECs matching bridge: 3,941

BRENDA protein-EC pairs (bridge-matched):
  Pairs:    28,894
  Proteins: 27,324
  ECs:      3,941


## 3. Rhea Catalytic Activity (comment_xml)

Rhea IDs and co-annotated EC numbers live in `comment_xml` as catalytic activity XML.
We extract the EC from the same XML block and use it to bridge to reactions.

In [5]:
rhea_raw = spark.sql("""
    SELECT entity_id, content
    FROM refdata_uniprot.comment_xml
    WHERE content LIKE '%Rhea%'
""").toPandas()

rhea_raw['protein'] = rhea_raw['entity_id'].str.replace('uniprot:', '', regex=False)

print(f'Rhea comment_xml rows: {len(rhea_raw):,}')
print(f'  Unique proteins: {rhea_raw.protein.nunique():,}')

Rhea comment_xml rows: 330,050
  Unique proteins: 236,103


In [6]:
ec_pat = re.compile(r'<dbReference\s+type="EC"\s+id="([^"]+)"')
rhea_pat = re.compile(r'<dbReference\s+type="Rhea"\s+id="(RHEA:\d+)"')

rhea_rows = []
rhea_only_count = 0
ec_only_count = 0
both_count = 0

for _, row in rhea_raw.iterrows():
    protein = row['protein']
    content = str(row['content'])
    ecs = ec_pat.findall(content)
    rheas = rhea_pat.findall(content)
    has_ec = len(ecs) > 0
    has_rhea = len(rheas) > 0
    if has_ec and has_rhea:
        both_count += 1
    elif has_rhea:
        rhea_only_count += 1
    elif has_ec:
        ec_only_count += 1
    for ec in ecs:
        rhea_rows.append({'protein': protein, 'ec': ec})

rhea_ec_df = pd.DataFrame(rhea_rows)
print(f'Rhea XML parsing:')
print(f'  Rows with both Rhea + EC: {both_count:,}')
print(f'  Rows with Rhea only:      {rhea_only_count:,}')
print(f'  Rows with EC only:        {ec_only_count:,}')
print(f'\nExtracted EC pairs from Rhea XML:')
print(f'  Protein-EC pairs: {len(rhea_ec_df):,}')
print(f'  Unique proteins:  {rhea_ec_df.protein.nunique():,}')
print(f'  Unique ECs:       {rhea_ec_df.ec.nunique():,}')

rhea_pairs = rhea_ec_df[rhea_ec_df['ec'].isin(bridge_ecs)][['protein', 'ec']].drop_duplicates()
rhea_pairs['channel'] = 'rhea'

print(f'\nRhea protein-EC pairs (bridge-matched):')
print(f'  Pairs:    {len(rhea_pairs):,}')
print(f'  Proteins: {rhea_pairs.protein.nunique():,}')
print(f'  ECs:      {rhea_pairs.ec.nunique():,}')

del rhea_raw, rhea_ec_df

Rhea XML parsing:
  Rows with both Rhea + EC: 257,232
  Rows with Rhea only:      72,818
  Rows with EC only:        0

Extracted EC pairs from Rhea XML:
  Protein-EC pairs: 257,232
  Unique proteins:  213,755
  Unique ECs:       5,119

Rhea protein-EC pairs (bridge-matched):
  Pairs:    192,345
  Proteins: 179,574
  ECs:      3,907


## 4. Combine Tier 1 Channels

Merge protein-EC pairs from all three channels. Track which channel(s) support each pair.

In [7]:
tier1_all = pd.concat([uniprot_ec_pairs, brenda_pairs, rhea_pairs], ignore_index=True)
print(f'All Tier 1 protein-EC-channel rows: {len(tier1_all):,}')

tier1_dedup = tier1_all.drop_duplicates(subset=['protein', 'ec', 'channel'])
print(f'After dedup: {len(tier1_dedup):,}')

tier1 = tier1_dedup.groupby(['protein', 'ec'])['channel'].agg(
    channels=lambda x: ','.join(sorted(set(x))),
    n_channels='nunique'
).reset_index()

print(f'\nDeduplicated Tier 1 protein-EC pairs: {len(tier1):,}')
print(f'  Unique proteins:  {tier1.protein.nunique():,}')
print(f'  Unique ECs:       {tier1.ec.nunique():,}')

All Tier 1 protein-EC-channel rows: 27,851,351


After dedup: 27,851,351



Deduplicated Tier 1 protein-EC pairs: 27,639,185


  Unique proteins:  26,549,024


  Unique ECs:       5,032


In [8]:
print(f'Channel support per protein-EC pair:')
for n in sorted(tier1['n_channels'].unique()):
    ct = (tier1['n_channels'] == n).sum()
    print(f'  {n} channel(s): {ct:,} pairs ({100*ct/len(tier1):.1f}%)')

print(f'\nChannel combination breakdown:')
combo = tier1['channels'].value_counts()
for ch, ct in combo.items():
    print(f'  {ch}: {ct:,}')

Channel support per protein-EC pair:
  1 channel(s): 27,438,863 pairs (99.3%)
  2 channel(s): 188,478 pairs (0.7%)
  3 channel(s): 11,844 pairs (0.0%)

Channel combination breakdown:


  uniprot_ec: 27,429,790


  rhea,uniprot_ec: 180,501
  brenda,rhea,uniprot_ec: 11,844
  brenda: 9,073
  brenda,uniprot_ec: 7,977


## 5. Coverage Summary

Compute reaction coverage via set intersection (no full merge needed).

In [9]:
balanced_ids = set(
    pd.read_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t', usecols=['id', 'status'])
    .query("status == 'OK'")['id']
    .str.replace('seed.reaction:', '', regex=False)
)

ec_ecs = set(uniprot_ec_pairs['ec'])
brenda_ec_set = set(brenda_pairs['ec'])
rhea_ec_set = set(rhea_pairs['ec'])
tier1_ec_set = set(tier1['ec'])

ec_rxns = set(ec_bridge[ec_bridge['ec'].isin(ec_ecs)]['rxn_bare'])
brenda_rxns = set(ec_bridge[ec_bridge['ec'].isin(brenda_ec_set)]['rxn_bare'])
rhea_rxns = set(ec_bridge[ec_bridge['ec'].isin(rhea_ec_set)]['rxn_bare'])
tier1_rxns = set(ec_bridge[ec_bridge['ec'].isin(tier1_ec_set)]['rxn_bare'])

print(f'Balanced reactions reached by Tier 1:')
print(f'  UniProt EC:  {len(ec_rxns):,} / {len(balanced_ids):,} ({100*len(ec_rxns)/len(balanced_ids):.1f}%)')
print(f'  BRENDA:      {len(brenda_rxns):,} / {len(balanced_ids):,} ({100*len(brenda_rxns)/len(balanced_ids):.1f}%)')
print(f'  Rhea:        {len(rhea_rxns):,} / {len(balanced_ids):,} ({100*len(rhea_rxns)/len(balanced_ids):.1f}%)')
print(f'  Combined:    {len(tier1_rxns):,} / {len(balanced_ids):,} ({100*len(tier1_rxns)/len(balanced_ids):.1f}%)')

print(f'\nProtein counts:')
print(f'  UniProt EC:  {uniprot_ec_pairs.protein.nunique():,}')
print(f'  BRENDA:      {brenda_pairs.protein.nunique():,}')
print(f'  Rhea:        {rhea_pairs.protein.nunique():,}')
print(f'  Combined:    {tier1.protein.nunique():,}')

print(f'\nAdditive value:')
print(f'  BRENDA beyond EC:      {len(brenda_rxns - ec_rxns):,} additional reactions')
print(f'  Rhea beyond EC:        {len(rhea_rxns - ec_rxns):,} additional reactions')
print(f'  Rhea beyond EC+BRENDA: {len(rhea_rxns - ec_rxns - brenda_rxns):,} additional reactions')

Balanced reactions reached by Tier 1:
  UniProt EC:  17,092 / 34,343 (49.8%)
  BRENDA:      12,239 / 34,343 (35.6%)
  Rhea:        12,129 / 34,343 (35.3%)
  Combined:    17,215 / 34,343 (50.1%)

Protein counts:


  UniProt EC:  26,543,162
  BRENDA:      27,324
  Rhea:        179,574


  Combined:    26,549,024

Additive value:
  BRENDA beyond EC:      123 additional reactions
  Rhea beyond EC:        0 additional reactions
  Rhea beyond EC+BRENDA: 0 additional reactions


In [10]:
tier1.to_parquet(f'{DATA_DIR}/uniprot_native_protein_ec.parquet', index=False)
print(f'Saved {len(tier1):,} protein-EC pairs to uniprot_native_protein_ec.parquet')
print(f'Columns: {list(tier1.columns)}')

Saved 27,639,185 protein-EC pairs to uniprot_native_protein_ec.parquet
Columns: ['protein', 'ec', 'channels', 'n_channels']


## 6. Summary

In [11]:
print('=' * 60)
print('NB03 TIER 1 EVIDENCE SUMMARY')
print('=' * 60)
print(f'\nChannels:')
print(f'  1. UniProt EC:  {uniprot_ec_pairs.protein.nunique():,} proteins, {uniprot_ec_pairs.ec.nunique():,} ECs -> {len(ec_rxns):,} reactions')
print(f'  2. BRENDA:      {brenda_pairs.protein.nunique():,} proteins, {brenda_pairs.ec.nunique():,} ECs -> {len(brenda_rxns):,} reactions')
print(f'  3. Rhea:        {rhea_pairs.protein.nunique():,} proteins, {rhea_pairs.ec.nunique():,} ECs -> {len(rhea_rxns):,} reactions')
print(f'\nCombined Tier 1:')
print(f'  {tier1.protein.nunique():,} proteins, {tier1.ec.nunique():,} ECs -> {len(tier1_rxns):,} / {len(balanced_ids):,} balanced reactions ({100*len(tier1_rxns)/len(balanced_ids):.1f}%)')
print(f'  {len(tier1):,} protein-EC pairs')
print(f'\nSaved: uniprot_native_protein_ec.parquet')
print(f'  (reaction expansion deferred to integration notebook)')
print(f'\nNext: NB04 -- pangenome annotation evidence (Tier 2)')

NB03 TIER 1 EVIDENCE SUMMARY

Channels:


  1. UniProt EC:  26,543,162 proteins, 4,938 ECs -> 17,092 reactions
  2. BRENDA:      27,324 proteins, 3,941 ECs -> 12,239 reactions
  3. Rhea:        179,574 proteins, 3,907 ECs -> 12,129 reactions

Combined Tier 1:


  26,549,024 proteins, 5,032 ECs -> 17,215 / 34,343 balanced reactions (50.1%)
  27,639,185 protein-EC pairs

Saved: uniprot_native_protein_ec.parquet
  (reaction expansion deferred to integration notebook)

Next: NB04 -- pangenome annotation evidence (Tier 2)
